# 🐍 Python od podstaw — Moduł 6: Programowanie obiektowe (klasy)

### Grupowanie danych i zachowań w jedną całość

W module 3 budowaliśmy inwentarz sklepu jako słownik słowników. Działało, ale przy
większej skali robi się niewygodne — trudno dodać walidację, trudno dodać metodę typu
„policz wartość” bez osobnej funkcji gdzieś obok. Klasy to sposób, żeby dane (atrybuty)
i operujące na nich funkcje (metody) trzymać razem, w jednym, samodzielnym bycie.

## Spis treści

1. [Po co programowanie obiektowe](#sec1)
2. [`class`, `__init__` i atrybuty](#sec2)
3. [Metody — parametr `self`](#sec3)
4. [Wiele instancji tej samej klasy](#sec4)
5. [Atrybuty klasy kontra atrybuty instancji](#sec5)
6. [`__str__` — ładne wypisywanie obiektu](#sec6)
7. [Konwencja `_` i `__` — enkapsulacja](#sec7)
8. [Dziedziczenie](#sec8)
9. [Nadpisywanie metod i polimorfizm](#sec9)
10. [Ciekawostka: w Pythonie dosłownie wszystko jest obiektem](#sec10)
11. [Podsumowanie modułu](#sec11)
12. [Ćwiczenia](#sec12)

---

<a id="sec1"></a>
## 1. Po co programowanie obiektowe

Przypomnij sobie ćwiczenie z inwentarzem z modułu 3:
`{"chleb": {"cena": 4.5, "ilosc": 20}}`. Żeby policzyć wartość jednego produktu, trzeba
było pamiętać, że to właśnie ten słownik ma klucze `cena` i `ilosc` — nic w kodzie tego
nie wymuszało, łatwo o literówkę czy pomyłkę. Klasa pozwala powiedzieć wprost: „produkt
to coś, co **zawsze** ma cenę i ilość, i **umie** policzyć swoją wartość” — dane i
logika stają się jedną, spójną całością.

<a id="sec2"></a>
## 2. `class`, `__init__` i atrybuty

Klasę definiuje się słowem `class`. Metoda `__init__` (tzw. **konstruktor**) wykonuje
się automatycznie przy tworzeniu nowego obiektu (instancji) i służy do ustawienia
początkowych **atrybutów** — danych przypisanych do konkretnego obiektu.

In [ ]:
class Osoba:
    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

kamil = Osoba("Kamil", 30)
print(kamil.imie)
print(kamil.wiek)
print(type(kamil))

> 💡 **Ciekawostka**
>
> `__init__` to jedna z tzw. metod «dunder» (od *double underscore* — podwójne podkreślenie). Python ma ich więcej (`__str__`, `__len__`, `__eq__`...) - to specjalne metody, które Python wywołuje automatycznie w konkretnych sytuacjach, np. przy tworzeniu obiektu, wypisywaniu go czy porównywaniu.

<a id="sec3"></a>
## 3. Metody — parametr `self`

Funkcja zdefiniowana wewnątrz klasy to **metoda**. Jej pierwszy parametr to zawsze
`self` — odniesienie do konkretnego obiektu, na którym metoda została wywołana. Nazwa
`self` to tylko **konwencja** (moglibyś nazwać ją inaczej), ale trzymaj się jej — cały
świat Pythona jej używa.

In [ ]:
class Osoba:
    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

    def przywitaj_sie(self):
        return f"Cześć, jestem {self.imie} i mam {self.wiek} lat"

    def czy_pelnoletnia(self):
        return self.wiek >= 18

kamil = Osoba("Kamil", 30)
print(kamil.przywitaj_sie())
print(kamil.czy_pelnoletnia())

> ⚠️ **Nie zapomnij o `self`**
>
> Brak `self` jako pierwszego parametru metody (`def przywitaj_sie():` zamiast `def przywitaj_sie(self):`) wywoła błąd przy wywołaniu `kamil.przywitaj_sie()` - Python automatycznie przekazuje obiekt jako pierwszy argument, więc metoda musi mieć na niego miejsce w liście parametrów.

<a id="sec4"></a>
## 4. Wiele instancji tej samej klasy

Klasa to **szablon** (przepis), a instancja to **konkretny obiekt** stworzony według
tego szablonu. Z jednej klasy można stworzyć dowolnie wiele niezależnych od siebie
instancji, każda z własnymi wartościami atrybutów.

In [ ]:
class Osoba:
    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

    def przywitaj_sie(self):
        return f"Cześć, jestem {self.imie} i mam {self.wiek} lat"

kamil = Osoba("Kamil", 30)
ania = Osoba("Ania", 25)

print(kamil.przywitaj_sie())
print(ania.przywitaj_sie())
print(kamil.imie == ania.imie)   # False - to dwa niezależne obiekty

<a id="sec5"></a>
## 5. Atrybuty klasy kontra atrybuty instancji

Atrybut zdefiniowany **wewnątrz `__init__`** (przez `self.coś = ...`) należy do
konkretnej instancji — każdy obiekt ma swoją własną wartość. Atrybut zdefiniowany
**bezpośrednio w ciele klasy** (poza metodami) jest współdzielony przez wszystkie
instancje tej klasy.

In [ ]:
class Osoba:
    gatunek = "Homo sapiens"   # atrybut klasy - wspólny dla wszystkich instancji

    def __init__(self, imie):
        self.imie = imie        # atrybut instancji - własny dla każdego obiektu

kamil = Osoba("Kamil")
ania = Osoba("Ania")

print(kamil.gatunek, ania.gatunek)   # to samo dla obu

Osoba.gatunek = "Homo sapiens sapiens"   # zmiana na poziomie klasy
print(kamil.gatunek, ania.gatunek)   # zmieniło się dla obu naraz

> 💡 **Typowe zastosowanie atrybutu klasy**
>
> Częsty wzorzec to licznik utworzonych obiektów - atrybut klasy startuje od 0, a każde wywołanie `__init__` go zwiększa (`Osoba.licznik += 1`). Zobaczysz to w ćwiczeniach.

<a id="sec6"></a>
## 6. `__str__` — ładne wypisywanie obiektu

Domyślnie `print(obiekt)` wypisuje coś mało czytelnego, w stylu
`<__main__.Osoba object at 0x...>`. Metoda `__str__` pozwala określić, co dokładnie ma
się wyświetlić — Python wywołuje ją automatycznie przy `print()` i `str()`.

In [ ]:
class Osoba:
    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

    def __str__(self):
        return f"Osoba: {self.imie} ({self.wiek} lat)"

kamil = Osoba("Kamil", 30)
print(kamil)          # bez __str__ zobaczylibyśmy <__main__.Osoba object at ...>
print(str(kamil))

<a id="sec7"></a>
## 7. Konwencja `_` i `__` — enkapsulacja

Python nie ma prawdziwie „prywatnych” atrybutów (jak np. Java), ale ma silną konwencję:
pojedyncze podkreślenie na początku (`self._atrybut`) oznacza „wewnętrzne, nie ruszaj z
zewnątrz”, a podwójne (`self.__atrybut`) włącza tzw. *name mangling* — utrudnia (choć
nie uniemożliwia) przypadkowy dostęp spoza klasy.

In [ ]:
class KontoBankowe:
    def __init__(self, saldo_poczatkowe):
        self._saldo = saldo_poczatkowe   # konwencja: "nie dotykaj bezpośrednio"

    def wplac(self, kwota):
        self._saldo += kwota

    def pokaz_saldo(self):
        return self._saldo

konto = KontoBankowe(1000)
konto.wplac(500)
print(konto.pokaz_saldo())

# Technicznie wciąż można to zrobić, ale to złamanie konwencji:
print(konto._saldo)

> ⚠️ **To konwencja, nie zamek**
>
> Pojedyncze podkreślenie niczego technicznie nie blokuje - `konto._saldo = 999999` zadziała bez błędu. To umowa między programistami: „to jest szczegół implementacji, zmieniaj przez metody (`wplac`, `wyplac`), nie bezpośrednio”. Łamanie tej konwencji nie jest błędem składniowym, ale jest złą praktyką.

<a id="sec8"></a>
## 8. Dziedziczenie

Klasa może **dziedziczyć** po innej — przejmuje wtedy wszystkie jej atrybuty i metody, i
może dodać własne albo nadpisać istniejące. Klasę, po której się dziedziczy, nazywamy
**bazową** (lub nadrzędną), a tę, która dziedziczy — **pochodną** (podklasą).

In [ ]:
class Zwierze:
    def __init__(self, imie):
        self.imie = imie

    def wydaj_dzwiek(self):
        return "..."

    def opisz(self):
        return f"{self.imie} mówi: {self.wydaj_dzwiek()}"


class Pies(Zwierze):   # Pies dziedziczy po Zwierze
    def wydaj_dzwiek(self):   # nadpisanie metody z klasy bazowej
        return "Hau hau!"


class Kot(Zwierze):
    def wydaj_dzwiek(self):
        return "Miau!"

burek = Pies("Burek")
mruczek = Kot("Mruczek")

print(burek.opisz())
print(mruczek.opisz())

<a id="sec9"></a>
## 9. Nadpisywanie metod i polimorfizm

Powyższy przykład to już **polimorfizm** — ta sama metoda `opisz()` z klasy bazowej
działa poprawnie dla każdej podklasy, mimo że `wydaj_dzwiek()` zachowuje się inaczej dla
każdej z nich. Gdy podklasa chce **rozszerzyć**, a nie całkiem zastąpić metodę bazową,
używa `super()`, żeby odwołać się do oryginalnej implementacji.

In [ ]:
class Zwierze:
    def __init__(self, imie):
        self.imie = imie

class Pies(Zwierze):
    def __init__(self, imie, rasa):
        super().__init__(imie)   # wywołanie __init__ z klasy bazowej
        self.rasa = rasa          # plus coś, czego Zwierze nie ma

burek = Pies("Burek", "Owczarek")
print(burek.imie, burek.rasa)

<a id="sec10"></a>
## 10. Ciekawostka: w Pythonie dosłownie wszystko jest obiektem

Liczby, stringi, funkcje, a nawet same klasy — wszystko to obiekty konkretnych klas.
`type()` (które znasz od modułu 1) tak naprawdę pokazuje klasę danej wartości.

In [ ]:
print(type(5))            # <class 'int'>
print(type("tekst"))      # <class 'str'>
print(type([1, 2, 3]))    # <class 'list'>
print(type(print))        # <class 'builtin_function_or_method'>

print(isinstance(5, int))     # sprawdzenie, czy obiekt jest instancją danej klasy
print(isinstance("a", int))

> 💡 **Ciekawostka**
>
> To dlatego w Pythonie na liczbach i stringach można wywoływać metody (`"tekst".upper()`, `(5).bit_length()`) - to nie są jakieś specjalne, wbudowane wyjątki od reguły, tylko zwykłe obiekty klas `str` i `int`, dokładnie tak samo jak Twoja klasa `Osoba`.

<a id="sec11"></a>
## 11. Podsumowanie modułu

Po tym module powinno być jasne:

- jak zdefiniować klasę (`class`), konstruktor (`__init__`) i metody,
- czym jest `self` i dlaczego musi być pierwszym parametrem metody,
- różnicę między atrybutem instancji a atrybutem klasy,
- jak działa `__str__` i po co jest,
- konwencję `_`/`__` jako sygnał „nie ruszaj bezpośrednio”,
- jak działa dziedziczenie, `super()` i nadpisywanie metod.

Programowanie obiektowe to duży temat — tu poznałeś/aś fundamenty, na których stoi
zdecydowana większość bibliotek Pythona (łącznie z tymi do analizy danych czy AI), więc
to solidna inwestycja na dalszą naukę.

<a id="sec12"></a>
## 12. Ćwiczenia

Kilka zadań świadomie wraca do wcześniejszych modułów (walidacja z `raise`, listy) —
tym razem opakowanych w klasy.

> 📝 **Ćwiczenie 1: Klasa Osoba**
>
> Zdefiniuj klasę `Osoba` z atrybutami `imie` i `zawod` (ustawianymi w `__init__`) oraz metodą `przedstaw_sie()`, zwracającą zdanie w stylu: „Cześć, jestem Kamil i pracuję jako programista”. Stwórz jedną instancję i wywołaj metodę.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
class Osoba:
    def __init__(self, imie, zawod):
        self.imie = imie
        self.zawod = zawod

    def przedstaw_sie(self):
        return f"Cześć, jestem {self.imie} i pracuję jako {self.zawod}"

kamil = Osoba("Kamil", "programista")
print(kamil.przedstaw_sie())
```
</details>

> 📝 **Ćwiczenie 2: Kilka instancji naraz**
>
> Zdefiniuj klasę `Pies` z atrybutem `imie` i metodą `szczekaj()`, zwracającą `f"{imie} mówi: Hau!"`. Stwórz trzy różne instancje (psy o różnych imionach) i wywołaj `szczekaj()` na każdym z nich w pętli `for`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
class Pies:
    def __init__(self, imie):
        self.imie = imie

    def szczekaj(self):
        return f"{self.imie} mówi: Hau!"

psy = [Pies("Burek"), Pies("Azor"), Pies("Reksio")]

for pies in psy:
    print(pies.szczekaj())
```
</details>

> 📝 **Ćwiczenie 3: Licznik instancji (atrybut klasy)**
>
> Zdefiniuj klasę `Uzytkownik` z atrybutem klasy `licznik = 0`, który zwiększa się o 1 w `__init__` za każdym razem, gdy tworzony jest nowy obiekt. Stwórz 3 instancje i wypisz `Uzytkownik.licznik` - powinno pokazać 3.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
class Uzytkownik:
    licznik = 0

    def __init__(self, nazwa):
        self.nazwa = nazwa
        Uzytkownik.licznik += 1

u1 = Uzytkownik("Kamil")
u2 = Uzytkownik("Ania")
u3 = Uzytkownik("Tomek")

print(Uzytkownik.licznik)   # 3
```

Podpowiedź: zmiana musi iść przez nazwę klasy (`Uzytkownik.licznik += 1`), nie przez `self.licznik += 1` - to drugie stworzyłoby nowy atrybut instancji, a nie zmieniłoby wspólny licznik.
</details>

> 📝 **Ćwiczenie 4: Ładne wypisywanie — `__str__`**
>
> Zdefiniuj klasę `Produkt` z atrybutami `nazwa` i `cena`, oraz metodą `__str__`, która zwraca np. `"Chleb - 4.50 zł"` (cena sformatowana do 2 miejsc po przecinku). Stwórz obiekt i wypisz go przez `print()`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
class Produkt:
    def __init__(self, nazwa, cena):
        self.nazwa = nazwa
        self.cena = cena

    def __str__(self):
        return f"{self.nazwa} - {self.cena:.2f} zł"

chleb = Produkt("Chleb", 4.5)
print(chleb)
```
</details>

> 📝 **Ćwiczenie 5: Konto bankowe z walidacją**
>
> Zdefiniuj klasę `KontoBankowe` z atrybutem `_saldo` (start od 0) oraz metodami `wplac(kwota)` i `wyplac(kwota)`. `wyplac` powinno zgłaszać `ValueError` (`raise`), jeśli próbujesz wypłacić więcej niż wynosi saldo. Przetestuj obie sytuacje - udaną wypłatę i tę, która powinna zgłosić błąd (w bloku `try`/`except`).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
class KontoBankowe:
    def __init__(self):
        self._saldo = 0

    def wplac(self, kwota):
        self._saldo += kwota

    def wyplac(self, kwota):
        if kwota > self._saldo:
            raise ValueError("Brak wystarczających środków na koncie")
        self._saldo -= kwota

konto = KontoBankowe()
konto.wplac(1000)
konto.wyplac(300)
print(konto._saldo)   # 700

try:
    konto.wyplac(10000)
except ValueError as blad:
    print(f"Błąd: {blad}")
```
</details>

> 📝 **Ćwiczenie 6: Dziedziczenie — kształty**
>
> Zdefiniuj klasę bazową `Ksztalt` z metodą `pole()`, zwracającą `0` (placeholder). Zdefiniuj podklasy `Prostokat(Ksztalt)` (z atrybutami `bok_a`, `bok_b`) i `Kolo(Ksztalt)` (z atrybutem `promien`), każda nadpisująca `pole()` własnym wzorem (dla koła: `3.14159 * promien ** 2`). Stwórz po jednej instancji i wypisz ich pola.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
class Ksztalt:
    def pole(self):
        return 0

class Prostokat(Ksztalt):
    def __init__(self, bok_a, bok_b):
        self.bok_a = bok_a
        self.bok_b = bok_b

    def pole(self):
        return self.bok_a * self.bok_b

class Kolo(Ksztalt):
    def __init__(self, promien):
        self.promien = promien

    def pole(self):
        return 3.14159 * self.promien ** 2

prostokat = Prostokat(4, 5)
kolo = Kolo(3)

print(f"Pole prostokąta: {prostokat.pole()}")
print(f"Pole koła: {kolo.pole():.2f}")
```
</details>

> 📝 **Ćwiczenie 7: Inwentarz jako klasa (refaktoryzacja modułu 3)**
>
> Zdefiniuj klasę `Produkt` (atrybuty `nazwa`, `cena`, `ilosc`, metoda `wartosc()` zwracająca `cena * ilosc`) oraz klasę `Inwentarz` z listą produktów (atrybut `produkty = []` w `__init__`), metodą `dodaj(produkt)` i metodą `wartosc_calkowita()`, sumującą `wartosc()` wszystkich produktów. Dodaj 3 produkty i wypisz łączną wartość.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
class Produkt:
    def __init__(self, nazwa, cena, ilosc):
        self.nazwa = nazwa
        self.cena = cena
        self.ilosc = ilosc

    def wartosc(self):
        return self.cena * self.ilosc

class Inwentarz:
    def __init__(self):
        self.produkty = []

    def dodaj(self, produkt):
        self.produkty.append(produkt)

    def wartosc_calkowita(self):
        suma = 0
        for produkt in self.produkty:
            suma += produkt.wartosc()
        return suma

inwentarz = Inwentarz()
inwentarz.dodaj(Produkt("Chleb", 4.5, 20))
inwentarz.dodaj(Produkt("Mleko", 3.2, 15))
inwentarz.dodaj(Produkt("Jajka", 12.0, 8))

print(f"Łączna wartość: {inwentarz.wartosc_calkowita():.2f} zł")
```

Zauważ, jak dużo czytelniejsze jest to podejście od słownika słowników z modułu 3 - `produkt.wartosc()` mówi wprost, co robi, a walidację (np. że cena nie może być ujemna) dałoby się łatwo dodać w jednym miejscu, w `__init__`.
</details>

> 🔥 **Ćwiczenie 8 (wyzwanie): Mini-system biblioteki**
>
> Zdefiniuj klasę `Ksiazka` (atrybuty `tytul`, `dostepna = True`) oraz klasę `Biblioteka` z listą książek i metodami: `dodaj_ksiazke(ksiazka)`, `wypozycz(tytul)` (znajduje książkę po tytule, ustawia `dostepna = False`, a jeśli książki nie ma albo jest już wypożyczona - zgłasza `ValueError` z odpowiednim komunikatem), oraz `pokaz_dostepne()` (wypisuje tytuły wszystkich dostępnych książek). Dodaj 3 książki, wypożycz jedną, spróbuj wypożyczyć ją ponownie (złap błąd), i na koniec pokaż dostępne.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
class Ksiazka:
    def __init__(self, tytul):
        self.tytul = tytul
        self.dostepna = True

class Biblioteka:
    def __init__(self):
        self.ksiazki = []

    def dodaj_ksiazke(self, ksiazka):
        self.ksiazki.append(ksiazka)

    def wypozycz(self, tytul):
        for ksiazka in self.ksiazki:
            if ksiazka.tytul == tytul:
                if not ksiazka.dostepna:
                    raise ValueError(f"'{tytul}' jest już wypożyczona")
                ksiazka.dostepna = False
                return
        raise ValueError(f"Nie znaleziono książki: '{tytul}'")

    def pokaz_dostepne(self):
        for ksiazka in self.ksiazki:
            if ksiazka.dostepna:
                print(ksiazka.tytul)

biblioteka = Biblioteka()
biblioteka.dodaj_ksiazke(Ksiazka("Wiedźmin"))
biblioteka.dodaj_ksiazke(Ksiazka("Lem: Solaris"))
biblioteka.dodaj_ksiazke(Ksiazka("Dune"))

biblioteka.wypozycz("Dune")

try:
    biblioteka.wypozycz("Dune")
except ValueError as blad:
    print(f"Błąd: {blad}")

print("Dostępne książki:")
biblioteka.pokaz_dostepne()
```

Podpowiedź: to naturalne połączenie klas (dane + zachowanie), list (kolekcja książek) i obsługi błędów z modułu 5 (`raise ValueError` przy próbie wypożyczenia niedostępnej pozycji) - wszystkie moduły tej serii spotykają się w jednym ćwiczeniu.
</details>

---

### Co dalej?

Jeśli mini-system biblioteki poszedł w miarę gładko, masz solidne podstawy
programowania obiektowego w Pythonie. Naturalne kolejne kroki: `dataclasses` (skrót na
pisanie prostych klas-kontenerów na dane), praca z zewnętrznymi bibliotekami (które w
większości same są zbudowane z klas), albo konkretny projekt łączący wszystkie sześć
modułów tej serii w jedną całość.